In [ ]:
from google.cloud import storage
import pandas as pd

# =============================================================================
# GCS MONTHLY-MEDIAN SHARD SIZE AUDIT
# =============================================================================

GCS_BUCKET = "bop-nca-data-space-s2-c1-staging"

YEARS = range(2016, 2026)

HISTORICAL_ROOT = (
    "monthly_medians/"
    "harmonic_feature_space_2016_2017"
)

NATIVE_ROOT = (
    "monthly_medians/"
    "harmonic_feature_space_2018_2025"
)


def root_for_year(year):
    return (
        HISTORICAL_ROOT
        if year <= 2017
        else NATIVE_ROOT
    )


def prefix_for_year(year):
    return (
        f"{root_for_year(year)}/"
        f"{year}/"
        f"S2SR_MonthlyMedians_{year}"
    )


client = storage.Client()

rows = []

for year in YEARS:

    prefix = prefix_for_year(year)

    blobs = [
        blob
        for blob in client.list_blobs(
            GCS_BUCKET,
            prefix=prefix,
        )
        if blob.name.lower().endswith(".tif")
    ]

    sizes = [
        int(blob.size or 0)
        for blob in blobs
    ]

    total_bytes = sum(sizes)

    rows.append({
        "year": year,
        "n_shards": len(blobs),
        "total_GB": total_bytes / 1e9,
        "total_GiB": total_bytes / (1024 ** 3),
        "mean_shard_GiB": (
            (total_bytes / len(blobs)) / (1024 ** 3)
            if blobs
            else 0
        ),
        "largest_shard_GiB": (
            max(sizes) / (1024 ** 3)
            if sizes
            else 0
        ),
        "gcs_prefix": (
            f"gs://{GCS_BUCKET}/{prefix}"
        ),
    })


size_table = pd.DataFrame(rows)

display(
    size_table[
        [
            "year",
            "n_shards",
            "total_GiB",
            "mean_shard_GiB",
            "largest_shard_GiB",
        ]
    ].round(3)
)


# =============================================================================
# TOTAL STORAGE / DOWNLOAD EXPECTATION
# =============================================================================

total_gib = size_table["total_GiB"].sum()

print(
    f"Total 2016–2025 shard volume: "
    f"{total_gib:.2f} GiB"
)

print(
    f"Mean per year: "
    f"{size_table['total_GiB'].mean():.2f} GiB"
)

print(
    f"Largest year: "
    f"{int(size_table.loc[size_table['total_GiB'].idxmax(), 'year'])} "
    f"({size_table['total_GiB'].max():.2f} GiB)"
)

,year,n_shards,total_GiB,mean_shard_GiB,largest_shard_GiB
0,2016,20,0.300,0.015,0.036
1,2017,20,0.296,0.015,0.036
2,2018,20,0.306,0.015,0.038
3,2019,20,0.309,0.015,0.039
4,2020,20,0.315,0.016,0.040
5,2021,20,0.311,0.016,0.039
6,2022,20,0.307,0.015,0.039
7,2023,20,0.312,0.016,0.039
8,2024,20,0.313,0.016,0.039
9,2025,20,0.311,0.016,0.039


Total 2016–2025 shard volume: 3.08 GiB
Mean per year: 0.31 GiB
Largest year: 2020 (0.32 GiB)


In [2]:
from google.cloud import storage
from pathlib import Path
import rasterio
import pandas as pd

YEAR = 2020

GCS_BUCKET = "bop-nca-data-space-s2-c1-staging"

ROOT = (
    "monthly_medians/"
    "harmonic_feature_space_2018_2025"
)

PREFIX = (
    f"{ROOT}/{YEAR}/"
    f"S2SR_MonthlyMedians_{YEAR}"
)

TMP_DIR = Path(
    rf"C:\NCA_DATA\scratch\median_shard_audit\{YEAR}"
)
TMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

client = storage.Client()

blobs = sorted(
    [
        b
        for b in client.list_blobs(
            GCS_BUCKET,
            prefix=PREFIX
        )
        if b.name.lower().endswith(".tif")
    ],
    key=lambda b: b.name
)

rows = []

for i, blob in enumerate(blobs):

    local = (
        TMP_DIR /
        Path(blob.name).name
    )

    # Only download the first shard for detailed raster inspection.
    if i == 0 and not local.exists():
        blob.download_to_filename(
            str(local)
        )

    row = {
        "file": Path(blob.name).name,
        "size_MiB": (blob.size or 0) / 1024**2
    }

    if i == 0:
        with rasterio.open(local) as src:

            row.update({
                "width": src.width,
                "height": src.height,
                "bands": src.count,
                "dtype": src.dtypes[0],
                "crs": str(src.crs),
                "res_x": abs(src.transform.a),
                "res_y": abs(src.transform.e),
                "left": src.bounds.left,
                "bottom": src.bounds.bottom,
                "right": src.bounds.right,
                "top": src.bounds.top,
                "nodata": src.nodata,
            })

    rows.append(row)

audit = pd.DataFrame(rows)

display(audit)

print(
    "\nTotal year size:",
    f"{audit['size_MiB'].sum()/1024:.3f} GiB"
)

print(
    "\nFirst shard detailed metadata:"
)

display(
    audit.iloc[[0]]
)

,file,size_MiB,width,height,bands,dtype,crs,res_x,res_y,left,bottom,right,top,nodata
0,S2SR_MonthlyMedians_20200000000000-0000000000.tif,30.133181,2560.0,2560.0,192.0,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,527740.0,4789810.0,553340.0,4815410.0,-32768.0
1,S2SR_MonthlyMedians_20200000000000-0000002560.tif,15.703857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,S2SR_MonthlyMedians_20200000000000-0000005120.tif,15.027667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,S2SR_MonthlyMedians_20200000000000-0000007680.tif,14.209320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S2SR_MonthlyMedians_20200000000000-0000010240.tif,1.795322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,S2SR_MonthlyMedians_20200000002560-0000000000.tif,22.063884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,S2SR_MonthlyMedians_20200000002560-0000002560.tif,22.472956,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,S2SR_MonthlyMedians_20200000002560-0000005120.tif,30.769461,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,S2SR_MonthlyMedians_20200000002560-0000007680.tif,36.463123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,S2SR_MonthlyMedians_20200000002560-0000010240.tif,1.808311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Total year size: 0.315 GiB

First shard detailed metadata:


,file,size_MiB,width,height,bands,dtype,crs,res_x,res_y,left,bottom,right,top,nodata
0,S2SR_MonthlyMedians_20200000000000-0000000000.tif,30.133181,2560.0,2560.0,192.0,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,527740.0,4789810.0,553340.0,4815410.0,-32768.0


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio


# =============================================================================
# SETTINGS
# =============================================================================

YEAR = 2020

SHARD_DIR = Path(
    rf"C:\NCA_DATA\scratch\S2_monthly_medians_stitch"
    rf"\gcs_shards\{YEAR}"
)

if not SHARD_DIR.is_absolute():
    raise ValueError("SHARD_DIR must be absolute.")

if not SHARD_DIR.exists():
    raise FileNotFoundError(SHARD_DIR)


MONTHS = [3, 7, 10]


# =============================================================================
# FILES
# =============================================================================

tifs = sorted(
    SHARD_DIR.glob("*.tif")
)

print(
    "Shards:",
    len(tifs)
)


# =============================================================================
# HELPERS
# =============================================================================

def get_band_index(src, name):

    descriptions = list(
        src.descriptions
    )

    if name not in descriptions:
        raise KeyError(
            f"{name} not found in {src.name}"
        )

    return descriptions.index(
        name
    ) + 1


def valid_mask(arr, nodata):

    valid = np.isfinite(
        arr
    )

    if nodata is not None:
        valid &= (
            arr != nodata
        )

    return valid


# =============================================================================
# AUDIT
# =============================================================================

rows = []


for month in MONTHS:

    mm = f"{month:02d}"

    total_pixels = 0

    rgb_valid_pixels = 0

    nobs_positive_pixels = 0

    nobs_values = []


    for path in tifs:

        with rasterio.open(
            path
        ) as src:

            idx_r = get_band_index(
                src,
                f"B4_M{mm}"
            )

            idx_g = get_band_index(
                src,
                f"B3_M{mm}"
            )

            idx_b = get_band_index(
                src,
                f"B2_M{mm}"
            )

            idx_nobs = get_band_index(
                src,
                f"nObs_M{mm}"
            )


            red = src.read(
                idx_r
            ).astype(
                "float32"
            )

            green = src.read(
                idx_g
            ).astype(
                "float32"
            )

            blue = src.read(
                idx_b
            ).astype(
                "float32"
            )

            nobs = src.read(
                idx_nobs
            ).astype(
                "float32"
            )


            r_valid = valid_mask(
                red,
                src.nodata
            )

            g_valid = valid_mask(
                green,
                src.nodata
            )

            b_valid = valid_mask(
                blue,
                src.nodata
            )

            rgb_valid = (
                r_valid
                & g_valid
                & b_valid
            )


            nobs_valid = valid_mask(
                nobs,
                src.nodata
            )

            positive = (
                nobs_valid
                & (nobs > 0)
            )


            total_pixels += (
                src.width
                * src.height
            )

            rgb_valid_pixels += int(
                rgb_valid.sum()
            )

            nobs_positive_pixels += int(
                positive.sum()
            )


            if positive.any():

                nobs_values.append(
                    nobs[
                        positive
                    ]
                )


    if nobs_values:

        x = np.concatenate(
            nobs_values
        )

    else:

        x = np.array(
            [],
            dtype=float
        )


    rows.append({

        "year":
            YEAR,

        "month":
            month,

        "total_grid_pixels":
            total_pixels,

        "rgb_valid_pixels":
            rgb_valid_pixels,

        "rgb_valid_fraction_grid":
            rgb_valid_pixels
            / total_pixels,

        "nObs_gt0_pixels":
            nobs_positive_pixels,

        "nObs_gt0_fraction_grid":
            nobs_positive_pixels
            / total_pixels,

        "nObs_p05":
            np.percentile(
                x,
                5
            )
            if len(x)
            else np.nan,

        "nObs_median":
            np.median(
                x
            )
            if len(x)
            else np.nan,

        "nObs_mean":
            np.mean(
                x
            )
            if len(x)
            else np.nan,

        "nObs_p95":
            np.percentile(
                x,
                95
            )
            if len(x)
            else np.nan,

        "nObs_max":
            np.max(
                x
            )
            if len(x)
            else np.nan,
    })


audit = pd.DataFrame(
    rows
)

display(
    audit
)

Shards: 20
